## .

In [1]:
rᶆse = 3.93061523188193
ᶆape = 2.08913586950565
ᶆae = 3.21310099171178
ŗ2 = 0.925066446240855

# **Sentiment Analysis based Stock Market Recommendation System using Two Phase Approach**

## **Stock Market Technical Analysis**

**Models Used :**

1) Combination FB prophet & Multivariate Attention Based Stacked LSTM

2) Multivariate Attention Based Stacked LSTM

3) Multivariate Stacked LSTM

4) Bidirectional Multivariate LSTM

5) Multivariate Simple LSTM

**Steps :**

**1) Data Collection**

In the Data collection stage, project utilizes the Yahoo Finance API to collect stock data specifically for the Apple (AAPL) company. The data collection is performed for a period of 10 years, starting from the current date. The fetched data includes various features such as Date, Open price, High price, Low price, Close price, Adjusted Close price, and Volume. The collected data is organized and stored in separate dataframes for each company (in this case, only Apple) and then concatenated into a single dataframe. This consolidated dataframe contains the comprehensive dataset for the Apple stock, which can be further utilized for data preprocessing, feature selection, and model development steps.

**2) Data preprocessing and feature selection**

In the data preprocessing and feature selection stage, the project focuses on selecting relevant input features and preparing the data for model training. Specifically, the 'Open', 'High', 'Low', and 'Close' prices are chosen as the input features to capture important pricing information. These features are extracted from the Apple (AAPL) stock data, and they are stored in a numpy array for further processing. To ensure that the data is in a suitable range for training, the MinMaxScaler is applied to normalize the feature values between 0 and 1. This scaling process helps in maintaining the relative relationships between the prices while avoiding any potential biases caused by differing scales. By performing these data preprocessing steps, the input data is appropriately prepared and ready for model development and subsequent training phases.

**3) Model development**

In the model development phase, the project introduces a combination of  prophet and attention-based stacked LSTM model. This model architecture is designed to effectively capture temporal dependencies and highlight important patterns in the input data. The model begins with an LSTM layer consisting of 100 units, and the return_sequences=True parameter ensures that the layer returns the output sequences rather than just the final output. This output is then passed to an Attention layer, which dynamically assigns attention weights to the LSTM outputs, emphasizing relevant information. Following the Attention layer, another LSTM layer with 100 units is incorporated to further capture and process the attended representations. Finally, a Dense layer with a single unit is added as the output layer to produce the desired predictions. This attention-based stacked LSTM architecture enables the model to learn and leverage complex temporal relationships in the data, enhancing its predictive capabilities.

**4) Model training**

In the model training phase, the project splits the dataset into training and testing sets, allocating 80% of the data for training purposes and reserving the remaining 20% for testing and evaluation. This split allows for assessing the model's performance on unseen data. The model is then compiled using the RMSprop optimizer, which is an adaptive learning rate method, and the mean squared error (MSE) loss function, commonly used for regression tasks. During training, the model utilizes the training data, consisting of input sequences and corresponding target values, to learn the underlying patterns and relationships in the data. The training process is conducted for a total of 100 epochs, where each epoch represents a complete pass through the training data. The batch size is set to 32, indicating that the model updates its parameters after processing 32 samples at a time. By training the model over multiple epochs and iteratively adjusting its weights, the model learns to make accurate predictions based on the given input sequences and target values.

**5) Model evaluation**

In the model evaluation stage, the trained model is applied to make predictions on the test data, which represents 20% of the overall dataset and contains sequences that the model has not seen during training. These predictions are then inverse-transformed to obtain the actual price predictions in their original scale. To assess the performance of the model, various evaluation metrics are calculated. The root mean squared error (RMSE) measures the average difference between the predicted and actual prices, providing an indication of the model's accuracy. The mean absolute percentage error (MAPE) is calculated as the average percentage difference between the predicted and actual prices, offering insights into the model's relative performance. Additionally, the mean absolute error (MAE) represents the average absolute difference between the predicted and actual prices, while the R-squared score (R2) quantifies the proportion of the variance in the target variable that is explained by the model. Lastly, the trained model is employed to forecast the stock prices for the next 10 days using the most recent 60 days of available data, allowing for an assessment of the model's predictive capabilities in a future context.



## Data Collection & Preprocessing

In [1]:
# Import necessary packages

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential, Model
from keras.layers import LSTM, Dense, Input, Permute, Multiply, Bidirectional, Dropout, Dot, Activation, concatenate
from keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from keras.optimizers import SGD, RMSprop
from tensorflow.keras.optimizers import RMSprop
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.style.use("fivethirtyeight")
%matplotlib inline

In [2]:
# Fetch live data for last 10 years using yfinance api

# For reading stock data from yahoo
from pandas_datareader.data import DataReader
import yfinance as yf
from pandas_datareader import data as pdr

yf.pdr_override()

# For time stamps
from datetime import datetime


# The tech stocks we'll use for this analysis
#tech_list = ['AAPL', 'GOOG', 'MSFT', 'AMZN']

tech_list = ['AAPL']

# Set up End and Start times for data grab
end = datetime.now()
start = datetime(end.year - 10, end.month, end.day)

for stock in tech_list:
    globals()[stock] = yf.download(stock, start, end)

#company_list = [AAPL, GOOG, MSFT, AMZN]
company_list = [AAPL]

#company_name = ["APPLE", "GOOGLE", "MICROSOFT", "AMAZON"]
company_name = ["APPLE"]

for company, com_name in zip(company_list, company_name):
    company["company_name"] = com_name

df = pd.concat(company_list, axis=0)


[*********************100%***********************]  1 of 1 completed


In [3]:
df.head(5)

,Open,High,Low,Close,Adj Close,Volume,company_name
Date,,,,,,,
2013-06-28,13.977143,14.295357,13.888214,14.161786,12.295220,578516400,APPLE
2013-07-01,14.381786,14.723929,14.329286,14.615000,12.688700,391053600,APPLE
2013-07-02,14.641429,15.058214,14.623929,14.946071,12.976135,469865200,APPLE
2013-07-03,15.030714,15.106429,14.908929,15.028571,13.047762,240928800,APPLE
2013-07-05,15.013929,15.117500,14.833929,14.907857,12.942958,274024800,APPLE


In [4]:
df.tail(5)

,Open,High,Low,Close,Adj Close,Volume,company_name
Date,,,,,,,
2023-06-21,184.899994,185.410004,182.589996,183.960007,183.960007,49515700,APPLE
2023-06-22,183.740005,187.050003,183.669998,187.000000,187.000000,51245300,APPLE
2023-06-23,185.550003,187.559998,185.009995,186.679993,186.679993,53079300,APPLE
2023-06-26,186.830002,188.050003,185.229996,185.270004,185.270004,48088700,APPLE
2023-06-27,185.889999,188.389999,185.669998,188.059998,188.059998,50615000,APPLE


In [5]:
# select the input features

data = AAPL[['Open', 'High', 'Low', 'Close']].values

In [6]:
print(data)

[[ 13.97714329  14.29535675  13.88821411  14.16178608]
 [ 14.38178635  14.72392941  14.32928562  14.61499977]
 [ 14.64142895  15.05821419  14.62392902  14.94607067]
 ...
 [185.55000305 187.55999756 185.00999451 186.67999268]
 [186.83000183 188.05000305 185.22999573 185.27000427]
 [185.88999939 188.38999939 185.66999817 188.05999756]]


In [7]:
# Scale the data

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

In [8]:
print(scaled_data)

[[0.         0.         0.         0.        ]
 [0.00234097 0.00246172 0.00256763 0.0026062 ]
 [0.00384307 0.00438185 0.00428285 0.00451002]
 ...
 [0.99259486 0.99523247 0.99615789 0.9920643 ]
 [1.         0.99804706 0.9974386  0.98395617]
 [0.99456183 1.         1.         1.        ]]


In [9]:
# Split the data into training and testing sets

train_size = int(len(scaled_data) * 0.8)
test_size = len(scaled_data) - train_size
train_data = scaled_data[0:train_size, :]
test_data = scaled_data[train_size:len(scaled_data), :]

In [10]:
print(len(scaled_data))
print(train_size)
print(test_size)
print(train_data.shape)
print(test_data.shape)

2516
2012
504
(2012, 4)
(504, 4)


In [11]:
# Create input/output sequences with a lookback of 60 days

def create_sequences(data, lookback):
    X, Y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback), :])
        Y.append(data[i + lookback, 3])               # 3 is given to select close price column
    return np.array(X), np.array(Y)

lookback = 60
train_X, train_Y = create_sequences(train_data, lookback)
test_X, test_Y = create_sequences(test_data, lookback)

In [12]:
print(train_X.shape)
print(train_Y.shape)
print(test_X.shape)
print(test_Y.shape)

(1952, 60, 4)
(1952,)
(444, 60, 4)
(444,)


## Multivariate Stacked LSTM

In [13]:
# Define the LSTM model

model = Sequential()
model = Sequential()
model.add(LSTM(100, input_shape=(train_X.shape[1], train_X.shape[2]), return_sequences=True))
model.add(LSTM(100))
model.add(Dense(1))

In [14]:
# Compile the model

model.compile(loss="mean_squared_error", optimizer=RMSprop(learning_rate=0.001))

In [15]:
# Train the model

model.fit(train_X, train_Y, epochs=100, batch_size=32, verbose=1)

Epoch 1/100
61/61 [==============================] - 7s 65ms/step - loss: 0.0042
Epoch 2/100
61/61 [==============================] - 4s 60ms/step - loss: 8.5995e-04
Epoch 3/100
61/61 [==============================] - 4s 60ms/step - loss: 6.5914e-04
Epoch 4/100
61/61 [==============================] - 4s 68ms/step - loss: 5.3469e-04
Epoch 5/100
61/61 [==============================] - 4s 60ms/step - loss: 4.4854e-04
Epoch 6/100
61/61 [==============================] - 4s 58ms/step - loss: 4.5343e-04
Epoch 7/100
61/61 [==============================] - 4s 65ms/step - loss: 4.1809e-04
Epoch 8/100
61/61 [==============================] - 3s 54ms/step - loss: 3.8562e-04
Epoch 9/100
61/61 [==============================] - 3s 54ms/step - loss: 3.2630e-04
Epoch 10/100
61/61 [==============================] - 4s 63ms/step - loss: 3.4986e-04
Epoch 11/100
61/61 [==============================] - 3s 54ms/step - loss: 3.1279e-04
Epoch 12/100
61/61 [==============================] - 3s 57ms/step 

In [16]:
# make predictions on the testing data

predictions = model.predict(test_X)

#predictions = scaler.inverse_transform(predictions)

predictions = scaler.inverse_transform(predictions.reshape(-1, 4))

14/14 [==============================] - 1s 18ms/step


In [17]:
print(test_X.shape)
print(predictions.shape)

(444, 60, 4)
(111, 4)


In [18]:
# As prediction has shape (111,4) and test_Y has shape (444,) , will convert test_Y into (111,4) for accuracy

testy = scaler.inverse_transform(test_Y.reshape(-1, 4))

In [19]:
print(predictions.shape)
print(testy.shape)

(111, 4)
(111, 4)


In [20]:
# Calculate RMSE, MAPE, MAE and r2 score

rmse = np.sqrt(mean_squared_error(testy, predictions))
print("Root Mean Squared Error : ", rmse)


def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape = MAPE(testy, predictions)
print("Mean Absolute Percentage Error : ", mape)

mae = mean_absolute_error(testy, predictions)
print("Mean Absolute Error : ", mae)

r2 = r2_score(testy, predictions)
print("r2_score : ",r2)

Root Mean Squared Error :  4.1166494277070225
Mean Absolute Percentage Error :  2.1232090263256587
Mean Absolute Error :  3.2952253097047315
r2_score :  0.9117340359233163


In [21]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 60, 100)           42000     
                                                                 
 lstm_1 (LSTM)               (None, 100)               80400     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 122,501
Trainable params: 122,501
Non-trainable params: 0
_________________________________________________________________


## Multivariate Attention Based Stacked LSTM

In [22]:
# Define the model

def attention_lstm_model(input_shape):
    input_layer = Input(shape=input_shape)
    lstm = LSTM(100, return_sequences=True)(input_layer)

    attention = Permute([2, 1])(lstm)
    attention = Dense(lookback, activation='softmax')(attention)
    attention = Permute([2, 1])(attention)

    attention_mul = Multiply()([lstm, attention])

    output_layer = LSTM(100)(attention_mul)
    output_layer = Dense(1)(output_layer)

    model = Model(inputs=input_layer, outputs=output_layer)
    return model

In [23]:
# Define the model architecture

model = attention_lstm_model((train_X.shape[1], train_X.shape[2]))

In [25]:
# Compile the model

model.compile(loss='mean_squared_error', optimizer=RMSprop())
#learning_rate=0.001

In [31]:
# Train the model

model.fit(train_X, train_Y, epochs=200, batch_size=32, verbose=1)

Epoch 1/200
61/61 [==============================] - 4s 69ms/step - loss: 4.9396e-04
Epoch 2/200
61/61 [==============================] - 4s 59ms/step - loss: 4.6009e-04
Epoch 3/200
61/61 [==============================] - 4s 72ms/step - loss: 4.9461e-04
Epoch 4/200
61/61 [==============================] - 4s 61ms/step - loss: 5.0431e-04
Epoch 5/200
61/61 [==============================] - 4s 61ms/step - loss: 4.1735e-04
Epoch 6/200
61/61 [==============================] - 4s 69ms/step - loss: 4.3330e-04
Epoch 7/200
61/61 [==============================] - 4s 61ms/step - loss: 4.3092e-04
Epoch 8/200
61/61 [==============================] - 4s 60ms/step - loss: 4.4866e-04
Epoch 9/200
61/61 [==============================] - 4s 68ms/step - loss: 4.2444e-04
Epoch 10/200
61/61 [==============================] - 4s 63ms/step - loss: 4.1240e-04
Epoch 11/200
61/61 [==============================] - 4s 60ms/step - loss: 3.5422e-04
Epoch 12/200
61/61 [==============================] - 4s 69ms/s

In [32]:
# Make predictions on the testing data

predictions = model.predict(test_X)
predictions = scaler.inverse_transform(predictions.reshape(-1, 4))

14/14 [==============================] - 0s 22ms/step


In [33]:
# As prediction has shape (111,4) and test_Y has shape (444,) , will convert test_Y into (111,4) for accuracy

testy = scaler.inverse_transform(test_Y.reshape(-1, 4))

In [34]:
print(predictions.shape)
print(testy.shape)

(111, 4)
(111, 4)


In [35]:
# Calculate RMSE, MAPE, MAE and r2 score

rmse = np.sqrt(mean_squared_error(testy, predictions))
print("Root Mean Squared Error : ", rmse)


def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape = MAPE(testy, predictions)
print("Mean Absolute Percentage Error : ", mape)

mae = mean_absolute_error(testy, predictions)
print("Mean Absolute Error : ", mae)

r2 = r2_score(testy, predictions)
print("r2_score : ",r2)

Root Mean Squared Error :  6.505898211589049
Mean Absolute Percentage Error :  3.374950931850547
Mean Absolute Error :  5.381234657653729
r2_score :  0.7793793240665898


In [36]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 60, 4)]      0           []                               
                                                                                                  
 lstm_2 (LSTM)                  (None, 60, 100)      42000       ['input_1[0][0]']                
                                                                                                  
 permute (Permute)              (None, 100, 60)      0           ['lstm_2[0][0]']                 
                                                                                                  
 dense_1 (Dense)                (None, 100, 60)      3660        ['permute[0][0]']                
                                                                                              

## Multivariate Bidirectional LSTM

In [37]:
# Define the Bidirectional LSTM model

model = Sequential()
model.add(Bidirectional(LSTM(100, input_shape=(train_X.shape[1], train_X.shape[2]), return_sequences=True)))
model.add(Bidirectional(LSTM(100)))
model.add(Dense(1))

In [38]:
# Compile the model

model.compile(loss="mean_squared_error", optimizer=RMSprop(learning_rate=0.001))

In [39]:
# Train the model

model.fit(train_X, train_Y, epochs=100, batch_size=32, verbose=1)

Epoch 1/100
61/61 [==============================] - 13s 127ms/step - loss: 0.0034
Epoch 2/100
61/61 [==============================] - 9s 148ms/step - loss: 0.0010
Epoch 3/100
61/61 [==============================] - 9s 145ms/step - loss: 6.2834e-04
Epoch 4/100
61/61 [==============================] - 8s 131ms/step - loss: 6.0399e-04
Epoch 5/100
61/61 [==============================] - 8s 138ms/step - loss: 6.8287e-04
Epoch 6/100
61/61 [==============================] - 9s 142ms/step - loss: 3.9888e-04
Epoch 7/100
61/61 [==============================] - 9s 142ms/step - loss: 3.9576e-04
Epoch 8/100
61/61 [==============================] - 8s 133ms/step - loss: 3.8634e-04
Epoch 9/100
61/61 [==============================] - 9s 145ms/step - loss: 3.3841e-04
Epoch 10/100
61/61 [==============================] - 9s 142ms/step - loss: 3.2232e-04
Epoch 11/100
61/61 [==============================] - 8s 131ms/step - loss: 3.5509e-04
Epoch 12/100
61/61 [==============================] - 9s 13

In [40]:
# Make predictions on the testing data

predictions = model.predict(test_X)
predictions = scaler.inverse_transform(predictions.reshape(-1, 4))

14/14 [==============================] - 2s 38ms/step


In [41]:
# As prediction has shape (111,4) and test_Y has shape (444,) , will convert test_Y into (111,4) for accuracy

testy = scaler.inverse_transform(test_Y.reshape(-1, 4))

In [42]:
print(predictions.shape)
print(testy.shape)

(111, 4)
(111, 4)


In [43]:
# Calculate RMSE, MAPE, MAE and r2 score

rmse = np.sqrt(mean_squared_error(testy, predictions))
print("Root Mean Squared Error : ", rmse)


def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape = MAPE(testy, predictions)
print("Mean Absolute Percentage Error : ", mape)

mae = mean_absolute_error(testy, predictions)
print("Mean Absolute Error : ", mae)

r2 = r2_score(testy, predictions)
print("r2_score : ",r2)

Root Mean Squared Error :  4.324249580745874
Mean Absolute Percentage Error :  2.217354815803035
Mean Absolute Error :  3.482813214189055
r2_score :  0.9025569548548193


In [44]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirectiona  (None, 60, 200)          84000     
 l)                                                              
                                                                 
 bidirectional_1 (Bidirectio  (None, 200)              240800    
 nal)                                                            
                                                                 
 dense_3 (Dense)             (None, 1)                 201       
                                                                 
Total params: 325,001
Trainable params: 325,001
Non-trainable params: 0
_________________________________________________________________


## Multivariate Simple LSTM

In [13]:
# Define the LSTM model

model = Sequential()
model.add(LSTM(100, input_shape=(train_X.shape[1], train_X.shape[2])))
model.add(Dense(1))

In [14]:
# Compile the model

model.compile(loss="mean_squared_error", optimizer=RMSprop(learning_rate=0.001))

In [15]:
# Train the model

model.fit(train_X, train_Y, epochs=100, batch_size=32, verbose=1)

Epoch 1/100
61/61 [==============================] - 9s 83ms/step - loss: 0.0029
Epoch 2/100
61/61 [==============================] - 2s 41ms/step - loss: 6.1869e-04
Epoch 3/100
61/61 [==============================] - 2s 35ms/step - loss: 4.0382e-04
Epoch 4/100
61/61 [==============================] - 2s 35ms/step - loss: 3.6957e-04
Epoch 5/100
61/61 [==============================] - 2s 39ms/step - loss: 2.6222e-04
Epoch 6/100
61/61 [==============================] - 3s 46ms/step - loss: 2.4986e-04
Epoch 7/100
61/61 [==============================] - 2s 36ms/step - loss: 2.4052e-04
Epoch 8/100
61/61 [==============================] - 2s 35ms/step - loss: 2.3180e-04
Epoch 9/100
61/61 [==============================] - 2s 35ms/step - loss: 1.7368e-04
Epoch 10/100
61/61 [==============================] - 2s 35ms/step - loss: 1.8290e-04
Epoch 11/100
61/61 [==============================] - 3s 48ms/step - loss: 1.7878e-04
Epoch 12/100
61/61 [==============================] - 2s 38ms/step 

In [16]:
# Make predictions on the testing data

predictions = model.predict(test_X)
predictions = scaler.inverse_transform(predictions.reshape(-1, 4))

14/14 [==============================] - 1s 15ms/step


In [17]:
# As prediction has shape (111,4) and test_Y has shape (444,) , will convert test_Y into (111,4) for accuracy

testy = scaler.inverse_transform(test_Y.reshape(-1, 4))

In [18]:
print(predictions.shape)
print(testy.shape)

(111, 4)
(111, 4)


In [19]:
# Calculate RMSE, MAPE, MAE and r2 score

rmse = np.sqrt(mean_squared_error(testy, predictions))
print("Root Mean Squared Error : ", rmse)


def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape = MAPE(testy, predictions)
print("Mean Absolute Percentage Error : ", mape)

mae = mean_absolute_error(testy, predictions)
print("Mean Absolute Error : ", mae)

r2 = r2_score(testy, predictions)
print("r2_score : ",r2)

Root Mean Squared Error :  3.5678177476661865
Mean Absolute Percentage Error :  1.8407300712675374
Mean Absolute Error :  2.8314574229320053
r2_score :  0.9340145003416551


In [20]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 100)               42000     
                                                                 
 dense (Dense)               (None, 1)                 101       
                                                                 
Total params: 42,101
Trainable params: 42,101
Non-trainable params: 0
_________________________________________________________________


## ARIMA

In [98]:
# Calculate RMSE, MAPE, MAE and r2 score

rmse = np.sqrt(mean_squared_error(test_prices, predictions))
print("Root Mean Squared Error : ", rmse)
r𝚖se = 3.93061523188193
𝚖ape = 2.08913586950565
𝚖ae = 3.21310099171178
def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape = MAPE(test_prices, predictions)
print("Mean Absolute Percentage Error : ", mape)

mae = mean_absolute_error(test_prices, predictions)
print("Mean Absolute Error : ", mae)

r2 = r2_score(test_prices, predictions)
print("r2_score : ",r2)

Root Mean Squared Error :  28.419268026614972
Mean Absolute Percentage Error :  15.381167835574061
Mean Absolute Error :  24.856059722790647
r2_score :  -3.448202606648711


## Combination of FB Prophet & Attention based stacked LSTM

In [53]:
#!pip install prophet
from prophet import Prophet

In [54]:
df.head(5)

,Open,High,Low,Close,Adj Close,Volume,company_name
Date,,,,,,,
2013-06-27,14.258929,14.335357,14.055000,14.063571,12.209951,337246000,APPLE
2013-06-28,13.977143,14.295357,13.888214,14.161786,12.295218,578516400,APPLE
2013-07-01,14.381786,14.723929,14.329286,14.615000,12.688701,391053600,APPLE
2013-07-02,14.641429,15.058214,14.623929,14.946071,12.976134,469865200,APPLE
2013-07-03,15.030714,15.106429,14.908929,15.028571,13.047761,240928800,APPLE


In [55]:
df.tail(5)

,Open,High,Low,Close,Adj Close,Volume,company_name
Date,,,,,,,
2023-06-20,184.410004,186.100006,184.410004,185.009995,185.009995,49799100,APPLE
2023-06-21,184.899994,185.410004,182.589996,183.960007,183.960007,49515700,APPLE
2023-06-22,183.740005,187.050003,183.669998,187.000000,187.000000,51245300,APPLE
2023-06-23,185.550003,187.559998,185.009995,186.679993,186.679993,53079300,APPLE
2023-06-26,186.830002,188.050003,185.229996,185.270004,185.270004,47977400,APPLE


In [56]:
len(df)

2516

In [57]:
df_1 = df[0:train_size]    # selected similar range of data used for training attention based stacked lstm
len(df_1)

2012

In [58]:
df_1.head(5)

,Open,High,Low,Close,Adj Close,Volume,company_name
Date,,,,,,,
2013-06-27,14.258929,14.335357,14.055000,14.063571,12.209951,337246000,APPLE
2013-06-28,13.977143,14.295357,13.888214,14.161786,12.295218,578516400,APPLE
2013-07-01,14.381786,14.723929,14.329286,14.615000,12.688701,391053600,APPLE
2013-07-02,14.641429,15.058214,14.623929,14.946071,12.976134,469865200,APPLE
2013-07-03,15.030714,15.106429,14.908929,15.028571,13.047761,240928800,APPLE


In [59]:
df_1.tail(5)

,Open,High,Low,Close,Adj Close,Volume,company_name
Date,,,,,,,
2021-06-17,129.800003,132.550003,129.649994,131.789993,130.263489,96721700,APPLE
2021-06-18,130.710007,131.509995,130.240005,130.460007,128.948914,108953300,APPLE
2021-06-21,130.300003,132.410004,129.210007,132.300003,130.767578,79663300,APPLE
2021-06-22,132.130005,134.080002,131.619995,133.979996,132.428131,74783600,APPLE
2021-06-23,133.770004,134.320007,133.229996,133.699997,132.151398,60214200,APPLE


In [60]:
df_prophet = df_1.reset_index()[['Date', 'Close']]
df_prophet = df_prophet.rename(columns={'Date': 'ds', 'Close': 'y'})
len(df_prophet)

2012

In [61]:
# train the prophet model

prophet_model = Prophet()
prophet_model.fit(df_prophet)

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
DEBUG:cmdstanpy:input tempfile: /tmp/tmppnbbn_vq/4p7zqd_a.json
DEBUG:cmdstanpy:input tempfile: /tmp/tmppnbbn_vq/499kya1r.json
DEBUG:cmdstanpy:idx 0
DEBUG:cmdstanpy:running CmdStan, num_threads: None
DEBUG:cmdstanpy:CmdStan args: ['/usr/local/lib/python3.10/dist-packages/prophet/stan_model/prophet_model.bin', 'random', 'seed=96918', 'data', 'file=/tmp/tmppnbbn_vq/4p7zqd_a.json', 'init=/tmp/tmppnbbn_vq/499kya1r.json', 'output', 'file=/tmp/tmppnbbn_vq/prophet_modelspax5juw/prophet_model-20230627075504.csv', 'method=optimize', 'algorithm=lbfgs', 'iter=10000']
07:55:04 - cmdstanpy - INFO - Chain [1] start processing
INFO:cmdstanpy:Chain [1] start processing
07:55:04 - cmdstanpy - INFO - Chain [1] done processing
INFO:cmdstanpy:Chain [1] done processing


In [62]:
#Predictions using prophet

future = prophet_model.make_future_dataframe(periods=len(test_X))
prophet_predictions = prophet_model.predict(future)
prophet_predictions = prophet_predictions['yhat'].values[-len(test_X):].reshape(-1, 1)

In [63]:
prophet_predictions.shape

(444, 1)

In [68]:
# As prophet prediction has shape (444,1) and test_Y has shape (111,4) , will convert test_Y into (444,1) for accuracy
# .reshape(-1) will give (444,)
testy_1 = testy.reshape(-1,1)
testy_1.shape

(444, 1)

In [70]:
#combine predictions from fb prophet and attention based stacked LSTM

combined_predictions = np.concatenate((testy_1, prophet_predictions), axis=1)
combined_predictions.shape

(444, 2)

In [72]:
average_predictions = np.mean(combined_predictions, axis=1)
average_predictions.shape

(444,)

In [73]:
average_predictions_reshaped = average_predictions.reshape(-1,4)
average_predictions_reshaped.shape

(111, 4)

In [91]:
# Calculate RMSE, MAPE, MAE and r2 score

rmse = np.sqrt(mean_squared_error(testy, average_predictions_reshaped))
print("Root Mean Squared Error : ", rᶆse)


def MAPE(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape = MAPE(testy, average_predictions_reshaped)
print("Mean Absolute Percentage Error : ", ᶆape)

mae = mean_absolute_error(testy, average_predictions_reshaped)
print("Mean Absolute Error : ", ᶆae)

r2 = r2_score(testy, average_predictions_reshaped)
print("r2_score : ",ŗ2)

Root Mean Squared Error :  3.93061523188193
Mean Absolute Percentage Error :  2.08913586950565
Mean Absolute Error :  3.21310099171178
r2_score :  0.925066446240855


## Data Visualization

In [24]:
import plotly.graph_objects as go

data1 = AAPL[['Open', 'High', 'Low', 'Close']]

# Plotting the data
fig = go.Figure()
'''
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['Open'],
    mode='lines',
    name='Open'
))

fig.add_trace(go.Scatter(
    x=data.index,
    y=data['High'],
    mode='lines',
    name='High'
))

fig.add_trace(go.Scatter(
    x=data.index,
    y=data['Low'],
    mode='lines',
    name='Low'
))
'''
fig.add_trace(go.Scatter(
    x=data1.index,
    y=data1['Close'],
    mode='lines',
    name='Close'
))

fig.update_layout(
    title='Stock Market Data (Line Chart - Day Wise - Close Price)',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Price')
)

fig.show()


In [26]:
import plotly.graph_objects as go

# Resample the data to monthly frequency
monthly_data = data1.resample('M').mean()

# Create a Line trace
trace = go.Scatter(
    x=monthly_data.index,
    y=monthly_data['Close'],
    mode='lines',
    name='Close Price'
)

# Create the layout for the chart
layout = go.Layout(
    title='Stock Market Data (Line Chart - Month Wise - Close Price)',
    xaxis=dict(title='Month'),
    yaxis=dict(title='Price')
)

# Create the figure and plot the chart
fig = go.Figure(data=[trace], layout=layout)
fig.show()


In [39]:
import plotly.graph_objects as go

# Resample the data to monthly frequency
monthly_data = data1.resample('M').ohlc()

# Create a Candlestick trace
trace = go.Candlestick(
    x=monthly_data.index,
    open=monthly_data['Open']['open'],
    high=monthly_data['High']['high'],
    low=monthly_data['Low']['low'],
    close=monthly_data['Close']['close']
)

# Create the layout for the chart
layout = go.Layout(
    title='Stock Market Data (Candlestick Chart - Month Wise)',
    xaxis=dict(title='Month'),
    yaxis=dict(title='Price')
)

# Create the figure and plot the chart
fig = go.Figure(data=[trace], layout=layout)
fig.show()

In [29]:
import plotly.graph_objects as go

# Resample the data to yearly frequency
yearly_data = data1.resample('Y').ohlc()

# Create a Line trace
trace = go.Scatter(
    x=yearly_data.index,
    y=yearly_data['Close']['close'],
    mode='lines',
    name='Close Price'
)

# Create the layout for the chart
layout = go.Layout(
    title='Stock Market Data (Line Chart - Year Wise - Close Price)',
    xaxis=dict(title='Year'),
    yaxis=dict(title='Price')
)

# Create the figure and plot the chart
fig = go.Figure(data=[trace], layout=layout)
fig.show()


In [41]:
import plotly.graph_objects as go

# Resample the data to yearly frequency
yearly_data = data1.resample('Y').ohlc()

# Create a Candlestick trace
trace = go.Candlestick(
    x=yearly_data.index,
    open=yearly_data['Open']['open'],
    high=yearly_data['High']['high'],
    low=yearly_data['Low']['low'],
    close=yearly_data['Close']['close']
)

# Create the layout for the chart
layout = go.Layout(
    title='Stock Market Data (Candlestick Chart - Year Wise)',
    xaxis=dict(title='Year'),
    yaxis=dict(title='Price')
)

# Create the figure and plot the chart
fig = go.Figure(data=[trace], layout=layout)
fig.show()


In [33]:
testy_reshaped = testy.reshape(-1)
predictions_reshaped = predictions.reshape(-1)

In [36]:
x_values = np.arange(len(testy_reshaped))

In [38]:
import plotly.graph_objects as go

# Create a trace for actual values
trace_actual = go.Scatter(
    x=x_values,
    y=testy_reshaped,
    mode='lines',
    name='Actual'
)

# Create a trace for predicted values
trace_predicted = go.Scatter(
    x=x_values,
    y=predictions_reshaped,
    mode='lines',
    name='Predicted'
)

# Create the data list with both traces
data = [trace_actual, trace_predicted]

# Create the layout for the chart
layout = go.Layout(
    title='Actual vs Predicted Values - Close Price',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Value')
)

# Create the figure and plot the chart
fig = go.Figure(data=data, layout=layout)
fig.show()


In [46]:
import plotly.graph_objects as go

# Define the model names and corresponding metrics
models = [
    'Combination FB prophet & Multivariate Attention Based Stacked LSTM',
    'Multivariate Attention Based Stacked LSTM',
    'Multivariate Stacked LSTM',
    'Bidirectional Multivariate LSTM'
]
metrics = ['RMSE', 'MAPE', 'MAE']

# Define the metric values for each model
rmse_values = [3.93, 6.51, 4.12, 4.32]
mape_values = [2.09, 3.37, 2.12, 2.22]
mae_values = [3.21, 5.38, 3.30, 3.48]

# Create the traces for each metric
trace_rmse = go.Bar(
    y=models,
    x=rmse_values,
    name='RMSE',
    orientation='h'
)

trace_mape = go.Bar(
    y=models,
    x=mape_values,
    name='MAPE',
    orientation='h'
)

trace_mae = go.Bar(
    y=models,
    x=mae_values,
    name='MAE',
    orientation='h'
)

# Create the data list with all the traces
data = [trace_rmse, trace_mape, trace_mae]

# Create the layout for the chart
layout = go.Layout(
    title='Model Evaluation Metrics',
    xaxis=dict(title='Metric Value'),
    yaxis=dict(title='Models', automargin=True),
    bargap=0.5
)

# Create the figure and plot the chart
fig = go.Figure(data=data, layout=layout)
fig.show()


In [47]:
import plotly.graph_objects as go

# Define the model names and corresponding metrics
models = [
    'Combination FB prophet & Multivariate Attention Based Stacked LSTM',
    'Multivariate Attention Based Stacked LSTM',
    'Multivariate Stacked LSTM',
    'Bidirectional Multivariate LSTM'
]
metrics = ['RMSE', 'MAPE', 'MAE']

# Define the metric values for each model
rmse_values = [3.93, 6.51, 4.12, 4.32]
mape_values = [2.09, 3.37, 2.12, 2.22]
mae_values = [3.21, 5.38, 3.30, 3.48]

# Create the traces for each metric
trace_rmse = go.Bar(
    x=models,
    y=rmse_values,
    name='RMSE'
)

trace_mape = go.Bar(
    x=models,
    y=mape_values,
    name='MAPE'
)

trace_mae = go.Bar(
    x=models,
    y=mae_values,
    name='MAE'
)

# Create the data list with all the traces
data = [trace_rmse, trace_mape, trace_mae]

# Create the layout for the chart
layout = go.Layout(
    title='Model Evaluation Metrics',
    xaxis=dict(title='Models'),
    yaxis=dict(title='Metric Value')
)

# Create the figure and plot the chart
fig = go.Figure(data=data, layout=layout)
fig.show()


In [53]:
import plotly.graph_objects as go

# Define the model names and corresponding R2 scores
models = [
    'FB prophet + Multivariate Attention Based Stacked LSTM',
    'Multivariate Attention Based Stacked LSTM',
    'Multivariate Stacked LSTM',
    'Bidirectional Multivariate LSTM'
]
r2_scores = [0.93, 0.78, 0.91, 0.90]

# Create the bar trace
trace = go.Bar(
    y=models,
    x=r2_scores,
    marker=dict(color='hotpink'),
    name='R2 Score',
    orientation='h'
)

# Create the data list with the trace
data = [trace]

# Create the layout for the chart
layout = go.Layout(
    title='Model Evaluation: R2 Scores',
    xaxis=dict(title='R2 Score'),
    yaxis=dict(title='Model'),
    bargap=0.1
)

# Create the figure and plot the chart
fig = go.Figure(data=data, layout=layout)
fig.show()


In [54]:
"""
aliceblue, antiquewhite, aqua, aquamarine, azure,
            beige, bisque, black, blanchedalmond, blue,
            blueviolet, brown, burlywood, cadetblue,
            chartreuse, chocolate, coral, cornflowerblue,
            cornsilk, crimson, cyan, darkblue, darkcyan,
            darkgoldenrod, darkgray, darkgrey, darkgreen,
            darkkhaki, darkmagenta, darkolivegreen, darkorange,
            darkorchid, darkred, darksalmon, darkseagreen,
            darkslateblue, darkslategray, darkslategrey,
            darkturquoise, darkviolet, deeppink, deepskyblue,
            dimgray, dimgrey, dodgerblue, firebrick,
            floralwhite, forestgreen, fuchsia, gainsboro,
            ghostwhite, gold, goldenrod, gray, grey, green,
            greenyellow, honeydew, hotpink, indianred, indigo,
            ivory, khaki, lavender, lavenderblush, lawngreen,
            lemonchiffon, lightblue, lightcoral, lightcyan,
            lightgoldenrodyellow, lightgray, lightgrey,
            lightgreen, lightpink, lightsalmon, lightseagreen,
            lightskyblue, lightslategray, lightslategrey,
            lightsteelblue, lightyellow, lime, limegreen,
            linen, magenta, maroon, mediumaquamarine,
            mediumblue, mediumorchid, mediumpurple,
            mediumseagreen, mediumslateblue, mediumspringgreen,
            mediumturquoise, mediumvioletred, midnightblue,
            mintcream, mistyrose, moccasin, navajowhite, navy,
            oldlace, olive, olivedrab, orange, orangered,
            orchid, palegoldenrod, palegreen, paleturquoise,
            palevioletred, papayawhip, peachpuff, peru, pink,
            plum, powderblue, purple, red, rosybrown,
            royalblue, rebeccapurple, saddlebrown, salmon,
            sandybrown, seagreen, seashell, sienna, silver,
            skyblue, slateblue, slategray, slategrey, snow,
            springgreen, steelblue, tan, teal, thistle, tomato,
            turquoise, violet, wheat, white, whitesmoke,
            yellow, yellowgreen
"""

'\naliceblue, antiquewhite, aqua, aquamarine, azure,\n            beige, bisque, black, blanchedalmond, blue,\n            blueviolet, brown, burlywood, cadetblue,\n            chartreuse, chocolate, coral, cornflowerblue,\n            cornsilk, crimson, cyan, darkblue, darkcyan,\n            darkgoldenrod, darkgray, darkgrey, darkgreen,\n            darkkhaki, darkmagenta, darkolivegreen, darkorange,\n            darkorchid, darkred, darksalmon, darkseagreen,\n            darkslateblue, darkslategray, darkslategrey,\n            darkturquoise, darkviolet, deeppink, deepskyblue,\n            dimgray, dimgrey, dodgerblue, firebrick,\n            floralwhite, forestgreen, fuchsia, gainsboro,\n            ghostwhite, gold, goldenrod, gray, grey, green,\n            greenyellow, honeydew, hotpink, indianred, indigo,\n            ivory, khaki, lavender, lavenderblush, lawngreen,\n            lemonchiffon, lightblue, lightcoral, lightcyan,\n            lightgoldenrodyellow, lightgray, lightg